# Phase 3: Evaluation & Anomaly Detection Results

---

## What This Notebook Does

In Phase 2, we trained an autoencoder for each machine type using **only normal sounds**.
Now we test whether these models can actually **detect anomalies** in the test set.

### The Test Set

Each machine type has 200 test files:
- **100 normal** sounds — the model should reconstruct these well (low MSE)
- **100 anomalous** sounds — the model should fail to reconstruct these (high MSE)

### How Anomaly Detection Works

```
Test .wav file
    ↓  librosa (same params as Phase 1)
Log-Mel Spectrogram
    ↓  saved scaler.transform()
Scaled spectrogram [0, 1]
    ↓  trained autoencoder
Reconstructed spectrogram
    ↓  MSE(original, reconstruction)
Anomaly Score  ← high = anomaly, low = normal
```

### Metrics We Compute

| Metric | What It Measures | Range |
|--------|-----------------|-------|
| **AUC-ROC** | How well the model separates normal from anomaly across ALL thresholds | 0.0 (worst) to 1.0 (perfect). 0.5 = random guessing. |
| **pAUC (max_fpr=0.1)** | AUC computed only at low false-positive rates (≤10%). The **DCASE primary metric**. | 0.0 to 1.0. Measures performance where it matters most — in the "high confidence" region. |

---

## 1. Library Imports

| Library | Purpose |
|---------|--------|
| **`librosa`** | Load test `.wav` files and compute Log-Mel Spectrograms (same as Phase 1) |
| **`joblib`** | Load the saved `MinMaxScaler` objects from Phase 1 |
| **`sklearn.metrics`** | Compute AUC-ROC and pAUC for anomaly detection evaluation |
| **`torch`** | Load trained models and perform inference |
| **`matplotlib`** | Visualize score distributions, ROC curves, and spectrogram reconstructions |

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os
import glob
import librosa
import joblib
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

---

## 2. Constants & Paths

We must use the **exact same** audio processing parameters as Phase 1. If we change any of
these (e.g., `N_MELS=64` instead of 128), the spectrograms will have different shapes and the
scaler/model will be incompatible.

In [ ]:
# Audio processing — MUST match Phase 1 exactly
SAMPLE_RATE = 16000
N_MELS      = 128
HOP_LENGTH  = 512
N_FFT       = 2048

# Paths
RAW_DIR       = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")
MODELS_DIR    = os.path.join("..", "models")
RESULTS_DIR   = os.path.join("..", "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

---

## 3. Redefine the Autoencoder Class

We need the **exact same** class definition to load the saved weights. This must match the
architecture from `02_training_v2.ipynb` exactly — same layers, same order, same parameter names.

> **Why not import it?** Jupyter notebooks don't natively support importing from other notebooks.
> In a production project you would put this class in a shared `.py` module. For now, we copy it.

In [ ]:
class AcousticAutoencoder(nn.Module):
    """
    Must match the architecture in 02_training_v2.ipynb exactly.
    """
    def __init__(self, input_dim, bottleneck_dim=32):
        super().__init__()
        self.input_dim = input_dim

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, bottleneck_dim),
            nn.ReLU(),
        )

        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),

            nn.Linear(512, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        original_shape = x.shape
        x = x.view(original_shape[0], -1)
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded.view(original_shape)


print("AcousticAutoencoder class defined.")

---

## 4. Evaluation Function

For each machine type, this function:

1. **Loads** the trained model and the MinMaxScaler from Phase 1
2. **Scans** the test directory for `.wav` files
3. **Extracts labels** from filenames: `"anomaly"` → 1, `"normal"` → 0
4. **Processes** each test file: load audio → spectrogram → scale → reconstruct → MSE
5. **Computes** AUC-ROC and pAUC (partial AUC at max FPR = 0.1)

### Label Extraction from Filenames

DCASE 2024 test files have labels embedded in their names:
```
section_00_source_test_anomaly_0001_n_A.wav  →  label = 1 (anomalous)
section_00_source_test_normal_0001_n_A.wav   →  label = 0 (normal)
section_00_target_test_anomaly_0001_n_X.wav  →  label = 1 (anomalous, target domain)
section_00_target_test_normal_0001_n_X.wav   →  label = 0 (normal, target domain)
```

### Per-Sample Anomaly Score

The anomaly score for a single test file is computed as:

$$\text{Score} = \text{MSE}(x, \hat{x}) = \frac{1}{n} \sum_{i=1}^{n} (x_i - \hat{x}_i)^2$$

Where $x$ is the scaled spectrogram and $\hat{x}$ is the model's reconstruction.

In [ ]:
def evaluate_machine(machine_name):
    """
    Evaluate a trained autoencoder on test data for one machine type.

    Returns a dict with scores, labels, AUC, pAUC, and file counts,
    plus the model instance for later visualization.
    """
    print(f"  Loading model and scaler...")

    # ---------------------------------------------------------------
    # 1. Load trained model
    # ---------------------------------------------------------------
    meta = torch.load(os.path.join(MODELS_DIR, machine_name, "metadata.pth"),
                      map_location=device, weights_only=False)
    model = AcousticAutoencoder(meta['input_dim'], meta['bottleneck_dim']).to(device)
    model.load_state_dict(
        torch.load(os.path.join(MODELS_DIR, machine_name, "best_model.pth"),
                   map_location=device, weights_only=True)
    )
    model.eval()  # Set to evaluation mode (disables Dropout, uses running BatchNorm stats)

    n_mels   = meta['n_mels']
    n_frames = meta['n_frames']

    # ---------------------------------------------------------------
    # 2. Load the scaler fitted on training data in Phase 1
    # ---------------------------------------------------------------
    scaler = joblib.load(os.path.join(PROCESSED_DIR, machine_name, "scaler.save"))

    # ---------------------------------------------------------------
    # 3. Find test .wav files
    # ---------------------------------------------------------------
    test_dir  = os.path.join(RAW_DIR, machine_name, "test")
    wav_files = sorted(glob.glob(os.path.join(test_dir, "*.wav")))

    if len(wav_files) == 0:
        print(f"  ⚠ No test files found in {test_dir}")
        return None

    print(f"  Found {len(wav_files)} test files")

    # ---------------------------------------------------------------
    # 4. Process each test file and compute anomaly scores
    # ---------------------------------------------------------------
    scores = []
    labels = []
    errors = 0

    for fpath in wav_files:
        fname = os.path.basename(fpath)

        # Extract label from filename
        if "anomaly" in fname:
            label = 1
        elif "normal" in fname:
            label = 0
        else:
            continue  # Skip files without a clear label

        try:
            # Load audio
            y, sr = librosa.load(fpath, sr=SAMPLE_RATE)

            # Compute Log-Mel Spectrogram (same params as Phase 1)
            mel_spec = librosa.feature.melspectrogram(
                y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS
            )
            log_mel = librosa.power_to_db(mel_spec, ref=np.max)

            # Verify shape matches training data
            if log_mel.shape != (n_mels, n_frames):
                # Pad or truncate to match expected width
                if log_mel.shape[1] > n_frames:
                    log_mel = log_mel[:, :n_frames]
                elif log_mel.shape[1] < n_frames:
                    pad_width = n_frames - log_mel.shape[1]
                    log_mel = np.pad(log_mel, ((0, 0), (0, pad_width)),
                                     mode='constant', constant_values=log_mel.min())

            # Scale using the training scaler
            H, W = log_mel.shape
            flat = log_mel.reshape(1, H * W)
            scaled = scaler.transform(flat)
            scaled = scaled.reshape(1, H, W)

            # Compute reconstruction error
            with torch.no_grad():
                x = torch.FloatTensor(scaled).to(device)
                recon = model(x)
                mse = torch.mean((recon - x) ** 2).item()

            scores.append(mse)
            labels.append(label)

        except Exception as e:
            errors += 1
            if errors <= 3:
                print(f"  ⚠ Error processing {fname}: {e}")

    if errors > 0:
        print(f"  ⚠ {errors} files had errors")

    scores = np.array(scores)
    labels = np.array(labels)

    # ---------------------------------------------------------------
    # 5. Compute metrics
    # ---------------------------------------------------------------
    auc  = roc_auc_score(labels, scores)
    pauc = roc_auc_score(labels, scores, max_fpr=0.1)

    n_normal  = int(np.sum(labels == 0))
    n_anomaly = int(np.sum(labels == 1))

    print(f"  Results: AUC={auc:.4f}  pAUC={pauc:.4f}  "
          f"({n_normal} normal, {n_anomaly} anomaly)")

    return {
        'scores': scores,
        'labels': labels,
        'auc': auc,
        'pauc': pauc,
        'n_normal': n_normal,
        'n_anomaly': n_anomaly,
        'model': model,
    }


print("Evaluation function defined.")

---

## 5. Run Evaluation for All Machine Types

We evaluate every machine that has both a trained model and test data.

In [ ]:
# Discover machine types with trained models
machine_types = sorted([
    d for d in os.listdir(MODELS_DIR)
    if os.path.isdir(os.path.join(MODELS_DIR, d))
    and os.path.exists(os.path.join(MODELS_DIR, d, "best_model.pth"))
])

print(f"Found {len(machine_types)} trained models: {machine_types}\n")

all_results = {}

for machine in machine_types:
    print(f"{'=' * 60}")
    print(f"EVALUATING: {machine}")
    print(f"{'=' * 60}")

    result = evaluate_machine(machine)
    if result is not None:
        all_results[machine] = result
    print()

---

## 6. Results Summary Table

### Understanding the Metrics

- **AUC = 1.0** → perfect separation between normal and anomaly (ideal)
- **AUC = 0.5** → no better than random guessing (model learned nothing useful)
- **AUC > 0.7** → acceptable for a baseline model
- **AUC > 0.8** → good performance
- **AUC > 0.9** → excellent performance

**pAUC** is harder to score high on because it only measures performance in the low
false-positive region — this is the metric that matters most in real industrial settings
where false alarms are costly.

In [ ]:
print(f"\n{'=' * 70}")
print(f"ANOMALY DETECTION RESULTS")
print(f"{'=' * 70}")
print(f"{'Machine':<12s} | {'AUC-ROC':>8s} | {'pAUC (10%)':>10s} | {'Normal':>6s} | {'Anomaly':>7s} | {'Verdict':>10s}")
print("-" * 70)

aucs = []
paucs = []

for machine in sorted(all_results.keys()):
    r = all_results[machine]
    auc = r['auc']
    pauc = r['pauc']
    aucs.append(auc)
    paucs.append(pauc)

    if auc >= 0.9:
        verdict = "Excellent"
    elif auc >= 0.8:
        verdict = "Good"
    elif auc >= 0.7:
        verdict = "Acceptable"
    elif auc >= 0.6:
        verdict = "Weak"
    else:
        verdict = "Poor"

    print(f"{machine:<12s} | {auc:8.4f} | {pauc:10.4f} | {r['n_normal']:6d} | {r['n_anomaly']:7d} | {verdict:>10s}")

print("-" * 70)
print(f"{'AVERAGE':<12s} | {np.mean(aucs):8.4f} | {np.mean(paucs):10.4f} |        |         |")
print(f"{'=' * 70}")

---

## 7. Anomaly Score Distributions

These histograms show the distribution of anomaly scores (MSE) for normal vs anomalous
test sounds. **Good separation** between the two distributions means the model can reliably
distinguish normal from anomalous.

- **Blue bars** = normal sounds (should cluster at LOW MSE)
- **Red bars** = anomalous sounds (should cluster at HIGH MSE)
- **More overlap** = harder to set a clean threshold = lower AUC

In [ ]:
n_machines = len(all_results)
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for idx, (machine, result) in enumerate(sorted(all_results.items())):
    if idx >= len(axes):
        break
    ax = axes[idx]

    normal_scores  = result['scores'][result['labels'] == 0]
    anomaly_scores = result['scores'][result['labels'] == 1]

    ax.hist(normal_scores, bins=30, alpha=0.6, color='steelblue', label='Normal', density=True)
    ax.hist(anomaly_scores, bins=30, alpha=0.6, color='crimson', label='Anomaly', density=True)

    ax.set_title(f"{machine} (AUC={result['auc']:.3f})", fontsize=11, fontweight='bold')
    ax.set_xlabel('Anomaly Score (MSE)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for idx in range(n_machines, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Anomaly Score Distributions (Normal vs Anomalous)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "score_distributions.png"), dpi=150, bbox_inches='tight')
print("Saved score distributions plot.")
plt.show()

---

## 8. ROC Curves

The **Receiver Operating Characteristic (ROC) curve** plots True Positive Rate vs False
Positive Rate at every possible threshold.

- A curve that hugs the **top-left corner** = excellent performance
- A curve along the **diagonal** = random guessing
- The **Area Under the Curve (AUC)** summarizes this into a single number

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for idx, (machine, result) in enumerate(sorted(all_results.items())):
    if idx >= len(axes):
        break
    ax = axes[idx]

    fpr, tpr, _ = roc_curve(result['labels'], result['scores'])
    ax.plot(fpr, tpr, color='steelblue', linewidth=2,
            label=f"AUC = {result['auc']:.3f}")
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')

    # Highlight the pAUC region (FPR ≤ 0.1)
    mask = fpr <= 0.1
    ax.fill_between(fpr[mask], 0, tpr[mask], alpha=0.2, color='orange',
                    label=f'pAUC region = {result["pauc"]:.3f}')

    ax.set_title(f"{machine}", fontsize=11, fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

for idx in range(n_machines, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('ROC Curves (per machine type)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "roc_curves.png"), dpi=150, bbox_inches='tight')
print("Saved ROC curves plot.")
plt.show()

---

## 9. Sample Reconstructions

Visual comparison of original vs. reconstructed spectrograms. For a well-trained model:

- **Normal sounds** should be reconstructed accurately (original and reconstruction look similar)
- **Anomalous sounds** should look noticeably different between original and reconstruction

We show one normal and one anomalous example for the first 4 machines.

In [ ]:
machines_to_show = sorted(all_results.keys())[:4]
fig, axes = plt.subplots(len(machines_to_show), 4, figsize=(18, 4 * len(machines_to_show)))

for row, machine in enumerate(machines_to_show):
    result = all_results[machine]
    model = result['model']
    model.eval()

    # Load scaler and metadata
    scaler = joblib.load(os.path.join(PROCESSED_DIR, machine, "scaler.save"))
    meta = torch.load(os.path.join(MODELS_DIR, machine, "metadata.pth"),
                      map_location=device, weights_only=False)
    n_mels, n_frames = meta['n_mels'], meta['n_frames']

    test_dir = os.path.join(RAW_DIR, machine, "test")

    # Find one normal and one anomaly file
    normal_file = sorted(glob.glob(os.path.join(test_dir, "*normal*.wav")))[0]
    anomaly_file = sorted(glob.glob(os.path.join(test_dir, "*anomaly*.wav")))[0]

    for col_offset, (fpath, title_prefix) in enumerate([(normal_file, 'Normal'), (anomaly_file, 'Anomaly')]):
        y, sr = librosa.load(fpath, sr=SAMPLE_RATE)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
        log_mel = librosa.power_to_db(mel, ref=np.max)

        if log_mel.shape[1] > n_frames:
            log_mel = log_mel[:, :n_frames]
        elif log_mel.shape[1] < n_frames:
            pad_w = n_frames - log_mel.shape[1]
            log_mel = np.pad(log_mel, ((0,0),(0,pad_w)), mode='constant', constant_values=log_mel.min())

        flat = log_mel.reshape(1, n_mels * n_frames)
        scaled = scaler.transform(flat).reshape(1, n_mels, n_frames)

        with torch.no_grad():
            x = torch.FloatTensor(scaled).to(device)
            recon = model(x).cpu().numpy()[0]
            mse = float(np.mean((scaled[0] - recon) ** 2))

        # Plot original
        ax_orig = axes[row, col_offset * 2]
        ax_orig.imshow(scaled[0], aspect='auto', origin='lower', cmap='magma')
        ax_orig.set_title(f"{machine} — {title_prefix} (Original)", fontsize=9)
        ax_orig.set_ylabel('Mel Band')

        # Plot reconstruction
        ax_recon = axes[row, col_offset * 2 + 1]
        ax_recon.imshow(recon, aspect='auto', origin='lower', cmap='magma')
        ax_recon.set_title(f"Reconstruction (MSE={mse:.5f})", fontsize=9)

plt.suptitle('Original vs Reconstruction — Normal & Anomalous Sounds', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "sample_reconstructions.png"), dpi=150, bbox_inches='tight')
print("Saved sample reconstructions plot.")
plt.show()

---

## 10. Final Summary

### What We Accomplished

| Step | Description |
|------|------------|
| Model Loading | Loaded trained autoencoders and scalers for all 7 machine types |
| Test Processing | Processed 200 test files per machine (100 normal + 100 anomalous) |
| Anomaly Scoring | Computed per-sample reconstruction error (MSE) as anomaly score |
| Metrics | Calculated AUC-ROC and pAUC for each machine type |
| Visualizations | Score distributions, ROC curves, sample reconstructions |

### Output Files

```
results/
├── score_distributions.png    ← Normal vs Anomaly score histograms
├── roc_curves.png             ← ROC curve per machine type
└── sample_reconstructions.png ← Original vs Reconstructed spectrograms
```

### Interpreting Your Results

- **AUC > 0.8** for most machines → your Dense Autoencoder baseline is working correctly
- **Some machines may score lower** (e.g., valve, slider) — these are known to be harder
  in the DCASE challenge due to subtler acoustic differences between normal and anomalous
- **pAUC scores** will be lower than AUC — this is expected. pAUC only measures the
  high-confidence region

### Potential Improvements

If results are unsatisfactory, consider:
1. **Switch to Convolutional Autoencoder** — exploits 2D spatial patterns in spectrograms
2. **Try BCE loss** instead of MSE — better gradient flow with Sigmoid outputs
3. **Adjust bottleneck size** — larger (64, 128) if underfitting, smaller (16) if overfitting
4. **Add data augmentation** — time/frequency masking (SpecAugment)
5. **Use Section-aware training** — separate domain A vs domain B files for Domain Shift